# Standard Visium cell2location (Figure 3 / S4)

Deconvolve infected and uninfected spleen Visium sections separately using matched scRNA-seq references (cell2location). Outputs per-spot cell-type abundances used for Fig. 3 / S4.

**Methods.** Raw UMIs; genes in ≥10 cells; genes shared with the reference; RegressionModel signatures → Cell2location (`N_cells_per_location=30`, `detection_alpha=20`).


## 1. Setup


In [ ]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
import scanpy as sc
import torch
import cell2location
import pyro.optim
from lightning.pytorch.callbacks import EarlyStopping

print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

# Paths (override with env vars or edit here)
VISIUM_PATH = Path(os.environ.get("VISIUM_ANNOTATED_H5AD", "data/visium_annotated.h5ad"))
SC_PATH = Path(os.environ.get("SC_DECONV_REF_H5AD", "data/sc_deconvolution_reference.h5ad"))
OUT_DIR = Path(os.environ.get("C2L_OUT", "outputs"))
OUT_DIR.mkdir(parents=True, exist_ok=True)

LABELS_KEY = "celltypes"
BATCH_KEY = "batch"


## 2. Load and split by condition

Visium is split on `condition` (infected / uninfected). The scRNA reference is split on `timepoint` (`3wk` / `0wk`) so each Visium set is mapped with a matched reference.


In [ ]:
def use_raw_counts(adata, layer_candidates):
    for layer in layer_candidates:
        if layer in adata.layers:
            adata.X = adata.layers[layer].astype(np.int32).copy()
            return adata
    X = adata.X.toarray() if hasattr(adata.X, "toarray") else np.asarray(adata.X)
    adata.X = np.round(X).astype(np.int32)
    return adata


def load_and_split(visium_path, sc_path):
    visium = sc.read_h5ad(visium_path)
    sc.pp.filter_genes(visium, min_cells=10)
    visium.var_names_make_unique()
    visium = use_raw_counts(visium, ["counts"])

    sc_ref = sc.read_h5ad(sc_path)
    sc.pp.filter_genes(sc_ref, min_cells=10)
    sc_ref.var_names_make_unique()
    sc_ref = use_raw_counts(sc_ref, ["raw_counts"])
    sc_ref.obs[BATCH_KEY] = pd.Categorical(sc_ref.obs[BATCH_KEY].astype(str))

    visium_u = visium[visium.obs["condition"] == "uninfected"].copy()
    visium_i = visium[visium.obs["condition"] == "infected"].copy()
    sc_u = sc_ref[sc_ref.obs["timepoint"] == "0wk"].copy()
    sc_i = sc_ref[sc_ref.obs["timepoint"] == "3wk"].copy()

    for ad in (sc_u, sc_i):
        ad.obs[BATCH_KEY] = pd.Categorical(ad.obs[BATCH_KEY].astype(str))

    return visium_u, visium_i, sc_u, sc_i


assert VISIUM_PATH.exists(), f"Missing Visium file: {VISIUM_PATH}"
assert SC_PATH.exists(), f"Missing scRNA reference: {SC_PATH}"

visium_uninfected, visium_infected, sc_uninfected, sc_infected = load_and_split(
    VISIUM_PATH, SC_PATH
)
print(
    "Visium infected/uninfected:",
    visium_infected.n_obs,
    visium_uninfected.n_obs,
)
print("scRNA infected/uninfected:", sc_infected.n_obs, sc_uninfected.n_obs)


## 3. Train cell2location

For each condition: fit reference signatures (`RegressionModel`), then map onto Visium (`Cell2location`). Abundance matrices are stored in `obsm['q05_cell_abundance_w_sf']` and `obsm['means_cell_abundance_w_sf']`.


In [ ]:
def clean_abundance_columns(adata):
    renames = {
        "q05_cell_abundance_w_sf": "q05cell_abundance_w_sf_means_per_cluster_mu_fg_",
        "means_cell_abundance_w_sf": "meanscell_abundance_w_sf_means_per_cluster_mu_fg_",
    }
    for key, prefix in renames.items():
        if key in adata.obsm:
            adata.obsm[key].columns = (
                adata.obsm[key].columns.astype(str).str.replace(prefix, "", regex=False)
            )
    return adata


def run_deconvolution(visium_data, sc_data, condition_name):
    cell2location.models.RegressionModel.setup_anndata(
        sc_data, batch_key=BATCH_KEY, labels_key=LABELS_KEY
    )
    ref_model = cell2location.models.RegressionModel(sc_data)
    ref_model.train(
        max_epochs=10000,
        accelerator="gpu" if torch.cuda.is_available() else "cpu",
        callbacks=[
            EarlyStopping(monitor="elbo_train", min_delta=0.001, patience=100, mode="min")
        ],
    )
    sc_data = ref_model.export_posterior(sc_data)

    shared = visium_data.var_names.intersection(sc_data.var_names)
    visium_data = visium_data[:, shared].copy()
    sc_data = sc_data[:, shared].copy()

    cell2location.models.Cell2location.setup_anndata(visium_data)
    model = cell2location.models.Cell2location(
        visium_data,
        cell_state_df=sc_data.varm["means_per_cluster_mu_fg"].loc[visium_data.var_names],
        N_cells_per_location=30,
        detection_alpha=20,
    )
    model.train(
        max_epochs=30000,
        accelerator="gpu" if torch.cuda.is_available() else "cpu",
        callbacks=[
            EarlyStopping(monitor="elbo_train", min_delta=0.001, patience=200, mode="min")
        ],
        plan_kwargs={
            "optim": pyro.optim.Adamax({"lr": 0.01, "weight_decay": 0.01}),
            "scale_elbo": 1.0,
        },
    )
    visium_data = model.export_posterior(visium_data)
    visium_data = clean_abundance_columns(visium_data)
    visium_data.obs["sample"] = visium_data.obs.index.str.split("_").str[4]
    visium_data.obs["condition"] = condition_name
    return visium_data


print("Training uninfected...")
visium_uninfected = run_deconvolution(visium_uninfected, sc_uninfected, "uninfected")
print("Training infected...")
visium_infected = run_deconvolution(visium_infected, sc_infected, "infected")


## 4. Save outputs


In [ ]:
out_u = OUT_DIR / "cell2location_uninfected_sd.h5ad"
out_i = OUT_DIR / "cell2location_infected_sd.h5ad"
visium_uninfected.write(out_u)
visium_infected.write(out_i)
print("Wrote:", out_u)
print("Wrote:", out_i)
print(
    "Abundance keys:",
    [k for k in visium_infected.obsm if "abundance" in k],
)
